# Boyce-Codd Normal Form (BCNF)

In notebook 04 the Library reached full normalization: it is in 1NF, 2NF,
and 3NF. BCNF is one more step stricter than 3NF. So why does this notebook
not touch the Library at all?

BCNF is only violated by a particular shape: a relation with **overlapping**
candidate keys where some functional dependency determines a prime attribute
from a non-superkey left side. The normalized Library has no table of that
shape so it is already in BCNF. There is nothing left to fix there.

To see a real BCNF violation we therefore use a small **constructed**
example. There is no database connection, no rebuild, and no SQL in this
notebook, just the `Relation` tool and a tiny hardcoded table for the visual.

In [ ]:
from rel_db_learning import Relation
from rel_db_learning.widgets.self_assessment import fd_check, multiple_choice
import pandas as pd

## What BCNF requires

**Boyce-Codd Normal Form (BCNF):** a relation is in BCNF if for **every**
non-trivial functional dependency `X -> Y`, the left side `X` is a superkey.

Compare this with 3NF. A relation is in 3NF if for every non-trivial FD
`X -> A`, either `X` is a superkey **or** `A` is a prime attribute (part of
some candidate key). BCNF drops that second option.

So BCNF is simply 3NF without the prime-attribute exception. Every relation
in BCNF is in 3NF, but not the other way around. The gap between them is
exactly the case where a non-superkey determines a prime attribute, which 3NF
tolerates and BCNF does not.

## A constructed example: addresses

Consider a relation `R(street, city, zip_code)` under two assumptions:

- A given street within a city has exactly one zip_code:
  `{street, city} -> zip_code`.
- A zip_code belongs to exactly one city: `zip_code -> city`.

These are simplifying assumptions chosen for teaching. Real postal systems are
more complex but this pair of dependencies produces the smallest clean example of
a relation that is in 3NF yet not in BCNF.

In [ ]:
addresses = pd.DataFrame(
    [
        ["Skalidi St", "Kissamos", "73400"],
        ["Irron Pol St", "Kissamos", "73400"],
        ["Omogenon St", "Kissamos", "73400"],
        ["Kissamou St", "Chania", "73100"],
        ["Apokoromou St", "Chania", "73100"],
        ["Chalepas St", "Chania", "73134"],
    ],
    columns=["street", "city", "zip_code"],
)
addresses

Notice the redundancy. Each zip_code always belongs to the same city: 73100
and 73134 both belong to Chania, and 73400 to Kissamos. The city is stored
once per street even though it is really a fact about the zip_code. Notice
also that a single city can have several zip_codes, so the city does not
determine the zip_code. If a zip_code were ever reassigned to a different
city we would have to update many rows to keep them consistent.

In [ ]:
address = Relation(["street", "city", "zip_code"], name="address")
address.add_fd(["street", "city"], ["zip_code"])
address.add_fd(["zip_code"], ["city"])

print("candidate_keys:", address.candidate_keys())
print("is_3nf:       ", address.is_3nf())
print("is_bcnf:      ", address.is_bcnf())
print("normal_form:  ", address.normal_form())

## Why this is 3NF but not BCNF

The tool reports two **overlapping** candidate keys, `{street, city}` and
`{street, zip_code}` both share `street`. Because of this both `city` and
`zip_code` are prime attributes.

The offending dependency is `zip_code -> city`:

- It **violates BCNF** because `zip_code` is not a superkey. Its closure is
  `{zip_code, city}`, which is not the whole relation.
- It still **satisfies 3NF** because the attribute it determines, `city`, is a
  prime attribute, so the 3NF prime-attribute exception applies.

This is exactly the 3NF but not BCNF case. The `zip_code -> city` redundancy
we saw in the table survives 3NF and is only removed by going to BCNF.

## Fixing it by BCNF decomposition

To reach BCNF we decompose the relation along the offending dependency. The
`decompose()` method does this for us and returns the resulting sub-relations,
each carrying the projection of the original dependencies onto its attributes.

In [ ]:
parts = address.decompose()

for part in parts:
    print(sorted(part.attributes), " is_bcnf:", part.is_bcnf())

print("lossless join:", address.is_lossless(parts))

## The trade-off: dependency preservation

The decomposition splits `address` into `{city, zip_code}` and
`{street, zip_code}`. Each part is in BCNF, and the join reconstructs the
original exactly, so the decomposition is **lossless**.

But notice what happened to `{street, city} -> zip_code`. Its attributes are
now spread across both tables, so that dependency can no longer be enforced
inside a single table. BCNF bought stricter redundancy removal at the cost of
**dependency preservation**.

This is a genuine trade-off. A BCNF decomposition is always lossless, but it
is not always dependency-preserving. A 3NF decomposition is always both. That
is why designers sometimes deliberately stop at 3NF rather than push to BCNF.

## The decomposition must be chosen correctly

Not every split is lossless. If we had instead decomposed on `city`, the join
would no longer reconstruct the original relation. The `is_lossless` check
catches this:

In [ ]:
address.is_lossless([
    Relation(["street", "city"]),
    Relation(["city", "zip_code"]),
])

This returns `False`. Splitting on `city` is lossy because `city` is not a key
for either side, so joining the two tables back together invents street and
zip_code combinations that were never in the original data.

## See it for yourself: lossless vs lossy

The `is_lossless` check gave you a True or False. The widget below shows you
why. It splits the same address table two ways and joins each split back
together, so you can see the result row by row.

Try both options:

- Split on zip_code: the join rebuilds the original table exactly. Nothing is
  added, nothing is lost. This is the lossless decomposition BCNF gave us.
- Split on city: the join invents street and zip_code combinations that were
  never in the original data. Those extra rows are highlighted. This is what
  "lossy" means in practice, and it is why the split on city returned False.

In [ ]:
from rel_db_learning.widgets import decomposition_explorer

decomposition_explorer()

## Check your understanding

Answer the questions below, then try the two exercises.

In [ ]:
multiple_choice(
    question="How does BCNF differ from 3NF?",
    options=[
        "BCNF requires atomic attribute values, while 3NF does not",
        "BCNF requires the left side of every non-trivial FD to be a superkey, "
        "while 3NF also allows the right side to be a prime attribute",
        "BCNF permits partial dependencies, while 3NF forbids them",
        "There is no difference; the two are equivalent",
    ],
    correct_index=1,
    explanation=(
        "3NF lets a non-superkey determine a prime attribute. BCNF removes that "
        "exception and demands a superkey on the left of every non-trivial FD, "
        "so BCNF is strictly stronger than 3NF."
    ),
)

In [ ]:
multiple_choice(
    question=(
        "In R(street, city, zip_code) with {street, city} -&gt; zip_code and "
        "zip_code -&gt; city, which dependency violates BCNF and why?"
    ),
    options=[
        "{street, city} -&gt; zip_code, because zip_code is non-prime",
        "zip_code -&gt; city, because zip_code is not a superkey",
        "zip_code -&gt; city, because city is non-prime",
        "Neither dependency violates BCNF",
    ],
    correct_index=1,
    explanation=(
        "The closure of zip_code is {zip_code, city}, not the whole relation, so "
        "zip_code is not a superkey and zip_code -&gt; city violates BCNF. It still "
        "satisfies 3NF because city is a prime attribute of the key {street, city}."
    ),
)

In [ ]:
multiple_choice(
    question=(
        "What can a BCNF decomposition sacrifice that a 3NF decomposition always "
        "preserves?"
    ),
    options=[
        "The lossless-join property",
        "Dependency preservation",
        "First normal form",
        "Atomic attribute values",
    ],
    correct_index=1,
    explanation=(
        "A BCNF decomposition is always lossless but can split a dependency across "
        "two tables so it is no longer enforceable in one table. 3NF always "
        "preserves dependencies, which is why designers sometimes stop at 3NF."
    ),
)

### Exercise 1: candidate keys

Here is another 3NF-but-not-BCNF relation, `R(a, b, c)` with `{a, b} -> c`
and `c -> a`. Work out its candidate keys by hand, fill them into
`student_keys` as a list of lists, and run the cell to check your answer.

In [ ]:
exercise = Relation(["a", "b", "c"], name="R")
exercise.add_fd(["a", "b"], ["c"])
exercise.add_fd(["c"], ["a"])

# Fill in your candidate keys, for example: [["a", "b"], ["a", "c"]]
student_keys = [
    # ["a", "b"],
]

fd_check(exercise, kind="candidate_keys", student_answer=student_keys)

### Exercise 2: decompose to BCNF

Now take the same relation `exercise` and bring it to BCNF. Decompose it,
confirm that each resulting part is in BCNF, and confirm that the
decomposition is lossless. The scaffold below does this; run it and read the
output.

In [ ]:
exercise_parts = exercise.decompose()

for part in exercise_parts:
    print(sorted(part.attributes), " is_bcnf:", part.is_bcnf())

print("lossless join:", exercise.is_lossless(exercise_parts))

## Closing

BCNF is the strictest normal form we cover. The Library reached full
normalization back in notebook 04, so here we stepped outside it to study the
rare 3NF-but-not-BCNF case: a relation with overlapping candidate keys and a
non-superkey determining a prime attribute. We saw how to remove that last
redundancy with a lossless decomposition, and why the cost in dependency
preservation sometimes makes 3NF the better stopping point.